# 🛍️ Customer Segmentation — Part 3: K-Means Clustering

**Input:** `data/rfm_scores.parquet` (from notebook 02)  
**Output:** `data/rfm_clustered.parquet` — RFM table with cluster assignments

---

## What we'll cover
1. Load RFM scores
2. Prepare & normalize features
3. Find optimal K — Elbow method + Silhouette score
4. Train final K-Means model
5. Visualize clusters in 2D and 3D
6. Profile each cluster
7. Compare clusters vs rule-based segments
8. Save clustered data


## 0. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, silhouette_samples
from sklearn.decomposition import PCA

import plotly.express as px
import plotly.graph_objects as go

warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   'white',
    'axes.spines.top':  False,
    'axes.spines.right':False,
    'axes.grid':        True,
    'grid.alpha':       0.3,
    'font.size':        12,
    'axes.titlesize':   14,
    'axes.titleweight': 'bold',
})

CLUSTER_COLORS = ['#1D9E75', '#378ADD', '#D85A30', '#7F77DD', '#BA7517']
RANDOM_STATE   = 42

print('Libraries loaded ✓')

## 1. Load RFM Data

In [ ]:
rfm = pd.read_parquet('../data/rfm_scores.parquet')

print(f'Customers : {len(rfm):,}')
print(f'Columns   : {rfm.columns.tolist()}')
rfm[['Recency','Frequency','Monetary']].describe().round(2)

## 2. Prepare Features for Clustering

We cluster on the **raw RFM values** (not the 1-5 scores) to preserve the full signal.  
We apply **log transformation** first to handle skewness, then **StandardScaler** to normalize.

In [ ]:
# ── Features ─────────────────────────────────────────────────────────────────
features = ['Recency', 'Frequency', 'Monetary']
X_raw    = rfm[features].copy()

# Log-transform to reduce skew (add 1 to avoid log(0))
X_log = np.log1p(X_raw)

# Standardize: mean=0, std=1
scaler  = StandardScaler()
X_scaled = scaler.fit_transform(X_log)
X_scaled = pd.DataFrame(X_scaled, columns=features)

# Visualize before vs after
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
fig.suptitle('Feature Distributions: Raw vs Scaled', fontsize=14, fontweight='bold')

colors = ['#1D9E75', '#378ADD', '#7F77DD']

for i, (col, color) in enumerate(zip(features, colors)):
    # Raw
    axes[0, i].hist(X_raw[col], bins=40, color=color, alpha=0.8, edgecolor='white')
    axes[0, i].set_title(f'Raw {col}')
    axes[0, i].set_ylabel('Customers')

    # Scaled
    axes[1, i].hist(X_scaled[col], bins=40, color=color, alpha=0.5, edgecolor='white')
    axes[1, i].set_title(f'Scaled {col}')
    axes[1, i].set_ylabel('Customers')

plt.tight_layout()
plt.savefig('../outputs/figures/feature_scaling.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Feature matrix shape: {X_scaled.shape}')

## 3. Find Optimal K

We use two complementary methods:
- **Elbow method** — plot inertia (within-cluster sum of squares) vs K
- **Silhouette score** — measures how well-separated clusters are (higher = better)

In [ ]:
K_RANGE    = range(2, 11)
inertias   = []
silhouettes = []

for k in K_RANGE:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels))
    print(f'K={k} | Inertia: {km.inertia_:,.1f} | Silhouette: {silhouette_score(X_scaled, labels):.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Finding Optimal Number of Clusters (K)', fontsize=14, fontweight='bold')

k_list = list(K_RANGE)

# Plot 1: Elbow curve
ax = axes[0]
ax.plot(k_list, inertias, 'o-', color='#1D9E75', linewidth=2.5, markersize=8)
ax.fill_between(k_list, inertias, alpha=0.1, color='#1D9E75')
ax.set_title('Elbow Method — Inertia')
ax.set_xlabel('Number of Clusters (K)')
ax.set_ylabel('Inertia (WCSS)')
ax.set_xticks(k_list)

# Plot 2: Silhouette scores
ax = axes[1]
bars = ax.bar(k_list, silhouettes, color='#378ADD', alpha=0.85, edgecolor='white')
ax.set_title('Silhouette Score per K')
ax.set_xlabel('Number of Clusters (K)')
ax.set_ylabel('Silhouette Score')
ax.set_xticks(k_list)

# Highlight best K
best_k_idx = np.argmax(silhouettes)
best_k     = k_list[best_k_idx]
bars[best_k_idx].set_color('#D85A30')
ax.annotate(f'Best K={best_k}',
            xy=(best_k, silhouettes[best_k_idx]),
            xytext=(best_k + 0.5, silhouettes[best_k_idx] + 0.01),
            fontsize=10, color='#D85A30', fontweight='bold')

plt.tight_layout()
plt.savefig('../outputs/figures/optimal_k.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nBest K by Silhouette: {best_k}')
print(f'Silhouette score    : {silhouettes[best_k_idx]:.4f}')

## 4. Train Final K-Means Model

In [ ]:
# ── Set K (use best from silhouette, adjust if elbow suggests otherwise) ──────
OPTIMAL_K = best_k  # override here if needed, e.g. OPTIMAL_K = 4

kmeans = KMeans(
    n_clusters  = OPTIMAL_K,
    random_state= RANDOM_STATE,
    n_init      = 20,        # more initializations = more stable result
    max_iter    = 500
)

rfm['Cluster'] = kmeans.fit_predict(X_scaled)

print(f'K-Means trained with K={OPTIMAL_K}')
print(f'Inertia         : {kmeans.inertia_:,.2f}')
print(f'Silhouette score: {silhouette_score(X_scaled, rfm["Cluster"]):.4f}')
print(f'\nCluster sizes:')
print(rfm['Cluster'].value_counts().sort_index())

In [ ]:
# ── Silhouette plot per cluster ───────────────────────────────────────────────
sil_vals   = silhouette_samples(X_scaled, rfm['Cluster'])
rfm['Silhouette'] = sil_vals

fig, ax = plt.subplots(figsize=(10, 6))
y_lower = 10

for i in range(OPTIMAL_K):
    cluster_sil = np.sort(sil_vals[rfm['Cluster'] == i])
    size_i = len(cluster_sil)
    y_upper = y_lower + size_i

    color = CLUSTER_COLORS[i % len(CLUSTER_COLORS)]
    ax.fill_betweenx(np.arange(y_lower, y_upper),
                     0, cluster_sil, alpha=0.75, color=color)
    ax.text(-0.05, y_lower + size_i / 2, f'C{i}', fontsize=10)
    y_lower = y_upper + 10

avg_sil = sil_vals.mean()
ax.axvline(avg_sil, color='red', linestyle='--', linewidth=1.5,
           label=f'Avg silhouette = {avg_sil:.3f}')
ax.set_title(f'Silhouette Plot — K={OPTIMAL_K}', fontweight='bold')
ax.set_xlabel('Silhouette coefficient')
ax.set_ylabel('Cluster')
ax.legend()
plt.tight_layout()
plt.savefig('../outputs/figures/silhouette_plot.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Visualize Clusters

In [ ]:
# ── 2D: PCA projection ────────────────────────────────────────────────────────
pca   = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_scaled)

rfm['PCA1'] = X_pca[:, 0]
rfm['PCA2'] = X_pca[:, 1]

explained = pca.explained_variance_ratio_

fig, ax = plt.subplots(figsize=(10, 7))

for i in range(OPTIMAL_K):
    mask = rfm['Cluster'] == i
    ax.scatter(
        rfm.loc[mask, 'PCA1'],
        rfm.loc[mask, 'PCA2'],
        s=25, alpha=0.5,
        color=CLUSTER_COLORS[i % len(CLUSTER_COLORS)],
        label=f'Cluster {i}'
    )

# Plot centroids
centroids_pca = pca.transform(kmeans.cluster_centers_)
ax.scatter(centroids_pca[:, 0], centroids_pca[:, 1],
           s=200, marker='X', color='black', zorder=5, label='Centroids')

ax.set_title(f'Customer Clusters — PCA Projection (K={OPTIMAL_K})', fontweight='bold')
ax.set_xlabel(f'PC1 ({explained[0]*100:.1f}% variance)')
ax.set_ylabel(f'PC2 ({explained[1]*100:.1f}% variance)')
ax.legend(fontsize=10)

plt.tight_layout()
plt.savefig('../outputs/figures/clusters_pca.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 3D interactive scatter: Recency, Frequency, Monetary ─────────────────────
fig_3d = px.scatter_3d(
    rfm,
    x='Recency',
    y='Frequency',
    z='Monetary',
    color='Cluster',
    color_discrete_sequence=CLUSTER_COLORS,
    opacity=0.5,
    title=f'Customer Clusters in RFM Space (K={OPTIMAL_K})',
    labels={
        'Recency':   'Recency (days)',
        'Frequency': 'Frequency (orders)',
        'Monetary':  'Monetary (£)',
        'Cluster':   'Cluster'
    },
    hover_data={'Customer ID': True}
)

fig_3d.update_traces(marker=dict(size=3))
fig_3d.update_layout(
    paper_bgcolor='white',
    plot_bgcolor='white',
    legend_title_text='Cluster'
)

fig_3d.write_html('../outputs/figures/clusters_3d.html')
fig_3d.show()
print('3D plot saved to outputs/figures/clusters_3d.html')

In [ ]:
# ── 2D pairplots: R vs F, R vs M, F vs M ─────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Cluster Separation across RFM Dimensions', fontsize=14, fontweight='bold')

pairs = [('Recency','Frequency'), ('Recency','Monetary'), ('Frequency','Monetary')]

for ax, (xcol, ycol) in zip(axes, pairs):
    for i in range(OPTIMAL_K):
        mask = rfm['Cluster'] == i
        ax.scatter(
            rfm.loc[mask, xcol],
            rfm.loc[mask, ycol],
            s=15, alpha=0.4,
            color=CLUSTER_COLORS[i % len(CLUSTER_COLORS)],
            label=f'C{i}'
        )
    ax.set_xlabel(xcol)
    ax.set_ylabel(ycol)
    ax.set_title(f'{xcol} vs {ycol}')
    ax.legend(fontsize=8, markerscale=2)

plt.tight_layout()
plt.savefig('../outputs/figures/clusters_pairplot.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Cluster Profiles

In [ ]:
# ── Cluster statistics ────────────────────────────────────────────────────────
cluster_profile = (
    rfm.groupby('Cluster')
    .agg(
        Customers     = ('Customer ID', 'count'),
        Avg_Recency   = ('Recency',    'mean'),
        Avg_Frequency = ('Frequency',  'mean'),
        Avg_Monetary  = ('Monetary',   'mean'),
        Total_Revenue = ('Monetary',   'sum'),
        Avg_R_Score   = ('R_Score',    'mean'),
        Avg_F_Score   = ('F_Score',    'mean'),
        Avg_M_Score   = ('M_Score',    'mean'),
    )
    .round(1)
)

total_rev = rfm['Monetary'].sum()
cluster_profile['Revenue_Share_%'] = (
    cluster_profile['Total_Revenue'] / total_rev * 100
).round(1)

cluster_profile = cluster_profile.sort_values('Avg_Monetary', ascending=False)
print(cluster_profile.to_string())

In [ ]:
# ── Spider/radar chart per cluster ───────────────────────────────────────────
# Normalize scores to 0-1 for radar
radar_cols = ['Avg_R_Score','Avg_F_Score','Avg_M_Score']
radar_labels = ['Recency', 'Frequency', 'Monetary']

fig, axes = plt.subplots(1, OPTIMAL_K, figsize=(4*OPTIMAL_K, 4),
                         subplot_kw=dict(polar=True))
fig.suptitle('Cluster RFM Profiles', fontsize=14, fontweight='bold')

angles = np.linspace(0, 2*np.pi, len(radar_labels), endpoint=False).tolist()
angles += angles[:1]  # close the circle

for idx, (cluster_id, row) in enumerate(cluster_profile.iterrows()):
    ax    = axes[idx] if OPTIMAL_K > 1 else axes
    vals  = [row['Avg_R_Score'], row['Avg_F_Score'], row['Avg_M_Score']]
    vals  += vals[:1]
    color = CLUSTER_COLORS[cluster_id % len(CLUSTER_COLORS)]

    ax.plot(angles, vals, color=color, linewidth=2)
    ax.fill(angles, vals, color=color, alpha=0.25)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(radar_labels, fontsize=10)
    ax.set_ylim(0, 5)
    ax.set_yticks([1, 2, 3, 4, 5])
    ax.set_yticklabels(['1','2','3','4','5'], fontsize=7)
    ax.set_title(f'Cluster {cluster_id}\n({int(row["Customers"]):,} customers)',
                 fontsize=10, pad=15)

plt.tight_layout()
plt.savefig('../outputs/figures/cluster_radar.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Box plots: RFM distributions per cluster ──────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('RFM Value Distributions by Cluster', fontsize=14, fontweight='bold')

for ax, col in zip(axes, ['Recency', 'Frequency', 'Monetary']):
    data_per_cluster = [rfm.loc[rfm['Cluster']==i, col].values
                        for i in range(OPTIMAL_K)]
    bp = ax.boxplot(data_per_cluster,
                    patch_artist=True,
                    medianprops=dict(color='black', linewidth=2))
    for patch, color in zip(bp['boxes'], CLUSTER_COLORS[:OPTIMAL_K]):
        patch.set_facecolor(color)
        patch.set_alpha(0.75)
    ax.set_title(col)
    ax.set_xlabel('Cluster')
    ax.set_xticklabels([f'C{i}' for i in range(OPTIMAL_K)])

plt.tight_layout()
plt.savefig('../outputs/figures/cluster_boxplots.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Compare Clusters vs Rule-Based Segments

In [ ]:
# Cross-tabulation: which rule-based segments fall into which clusters?
crosstab = pd.crosstab(
    rfm['Segment_RuleBased'],
    rfm['Cluster'],
    margins=True
)

print('Rule-based segments vs K-Means clusters:')
print(crosstab.to_string())

In [ ]:
# Heatmap of cross-tabulation (normalized by row)
crosstab_pct = pd.crosstab(
    rfm['Segment_RuleBased'],
    rfm['Cluster'],
    normalize='index'
).round(2)

fig, ax = plt.subplots(figsize=(10, 7))
sns.heatmap(
    crosstab_pct,
    annot=True, fmt='.0%',
    cmap='YlGn',
    linewidths=0.5,
    ax=ax,
    cbar_kws={'label': 'Proportion of segment in cluster'}
)
ax.set_title('Rule-Based Segments vs K-Means Clusters', fontweight='bold')
ax.set_xlabel('K-Means Cluster')
ax.set_ylabel('Rule-Based Segment')
plt.tight_layout()
plt.savefig('../outputs/figures/cluster_vs_rulebased.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Save Clustered Data

In [ ]:
import pickle, os
os.makedirs('../outputs/figures', exist_ok=True)
os.makedirs('../outputs/models',  exist_ok=True)

# Save clustered RFM table
rfm.to_parquet('../data/rfm_clustered.parquet', index=False)
rfm.to_csv('../outputs/rfm_clustered.csv', index=False)

# Save model + scaler for reuse
with open('../outputs/models/kmeans_model.pkl', 'wb') as f:
    pickle.dump(kmeans, f)
with open('../outputs/models/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print('Saved:')
print(f'  data/rfm_clustered.parquet')
print(f'  outputs/rfm_clustered.csv')
print(f'  outputs/models/kmeans_model.pkl')
print(f'  outputs/models/scaler.pkl')
print(f'\nFinal dataset shape: {rfm.shape}')
rfm.head()

## ✅ Clustering Summary

| Step | Detail |
|------|--------|
| **Features** | Recency, Frequency, Monetary (log-transformed + standardized) |
| **Optimal K** | Determined by Silhouette score |
| **Model** | K-Means (n_init=20 for stability) |
| **Evaluation** | Elbow + Silhouette plot |
| **Visualization** | PCA 2D, interactive 3D, pairplot, radar charts |

**➡️ Next: Notebook 04 — Segment Insights & Marketing Strategy**